In [1]:
from __future__ import annotations

from pathlib import Path
import dataretrieval.nwis as nwis
from datetime import datetime
import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pynhd as nhd
from pynhd import NLDI, NHDPlusHR, WaterData
import py3dep
import pygeohydro as gh

from pathlib import Path
import networkx as nx
import xarray as xr
import xrspatial
import os

In [2]:
nldi = NLDI()
station_id = "09010500" # NWIS id for Tuolumne river at the mouth of Hetch Hetchy Reservoir
basinname = 'ColoradoRiverBasin'
#Getting basin geometry
basin = nldi.get_basins(station_id)
geometry = basin.geometry.iloc[0]


In [7]:
topo = py3dep.get_map(["DEM", "Slope Degrees"], geometry, 90, geo_crs=4326, crs=5070) # Get the DEM and slope for the basin geometry at 90m resolution, reproject to 5070, and convert slope from degrees to m/m
dem = py3dep.get_dem(geometry, 30) # Get the DEM for the basin geometry at 30m resolution
dem = dem.rio.reproject(5070) # Reproject the DEM to match the CRS of the slope and the flowlines
slope = py3dep.deg2mpm(xrspatial.slope(dem)) # Calculate slope in m/m from the DEM using xrspatial, which handles the spatial resolution and units correctly
topo = xr.merge([dem, slope]) # Merge the DEM and slope into a single xarray dataset


# if not os.path.exists('files/DEM'):
#     os.makedirs('files/DEM')
# dem.rio.to_raster(Path("files/DEM", f"dem_{station_id}.tif"))

In [9]:
if not os.path.exists('files/DEM'):
    os.makedirs('files/DEM')
dem.rio.to_raster(Path("files/DEM", f"dem_{station_id}.tif"))

In [8]:
# build a dataframe to get record basin information
#get a range of elevation within the basin
ave_basin_elevation = topo.elevation.mean().values
min_basin_elevation = topo.elevation.min().values
max_basin_elevation = topo.elevation.max().values

#get the average slope of the basin
ave_basin_slope = topo.slope.mean().values

#get the basin area
gs = gpd.GeoSeries([geometry], crs="EPSG:4326")

# Project to a metric system (EPSG:3857 is Web Mercator / meters)
gs_meters = gs.to_crs(epsg=3857)

# Calculate area (returns square meters)
area_m2 = gs_meters.area.iloc[0]
area_km2 = area_m2 / 1_000_000

#Put the informatininto  a dataframe
basin_info = pd.DataFrame({
    "Basin_Name": [basinname],
    "station_id": [station_id],
    "Average_Elevation_m": [ave_basin_elevation],
    "Minimum_Elevation_m": [min_basin_elevation],
    "Maximum_Elevation_m": [max_basin_elevation],
    "Average_Slope": [ave_basin_slope],
    "Area_km2": [area_km2],
})

gdf = gpd.GeoDataFrame(index=[0], crs="EPSG:4326", geometry=[geometry])

# Use this gdf in your nlcd_bygeom call nlcd is the National Land Cover Database, and nlcd_bygeom is a function in pygeohydro that retrieves the land cover types within a given geometry
# nlcd_data = gh.nlcd_bygeom(gdf)

# station_ids = [station_id]
# #geometry = NLDI().get_basins(station_ids)
# basin.index = station_ids

Getting Land Information

In [10]:
# cover and impervious descriptor data at a 100 m resolution for all three stations
desc = gh.nlcd_bygeom(basin.geometry, 100, years={"descriptor": 2019}, ssl=False)
#land cover data at a 100 m resolution for all three stations for 2016 and 2019
lulc = gh.nlcd_bygeom(basin.geometry, 100, years={"cover": [2016, 2019]}, ssl=False)


In [11]:
stats = gh.cover_statistics(lulc[f"USGS-{station_id}"].cover_2019) # Get the land cover statistics for the 2019 land cover data for the station's basin geometry
roughness = gh.overland_roughness(lulc[f"USGS-{station_id}"].cover_2019) # Get the overland flow roughness for the 2019 land cover data for the station's basin geometry

In [12]:
cmap, norm, levels = gh.plot.cover_legends() # Get the colormap, normalization, and levels for plotting land cover data using pygeohydro's built-in legends
cover = lulc[f"USGS-{station_id}"].cover_2019 # Get the 2019 land cover data for the station's basin geometry
unique_lulc = np.unique(cover.values) # Get the unique land cover codes present in the cover data for the station's basin geometry

nlcd_legend = {
    11: "Open Water",
    12: "Perennial Ice/Snow",
    21: "Developed, Open Space",
    22: "Developed, Low Intensity",
    23: "Developed, Medium Intensity",
    24: "Developed, High Intensity",
    31: "Barren Land (Rock/Sand/Clay)",
    41: "Deciduous Forest",
    42: "Evergreen Forest",
    43: "Mixed Forest",
    52: "Shrub/Scrub",
    71: "Grassland/Herbaceous",
    90: "Woody Wetlands",
    95: "Emergent Herbaceous Wetlands",
    127: "No Data / Masked"  # Often used as a fill or background value
}

# To translate your list of unique values:
unique_names = [nlcd_legend.get(code, "Unknown") for code in unique_lulc]


unique_names


['Open Water',
 'Perennial Ice/Snow',
 'Developed, Open Space',
 'Developed, Low Intensity',
 'Developed, Medium Intensity',
 'Developed, High Intensity',
 'Barren Land (Rock/Sand/Clay)',
 'Deciduous Forest',
 'Evergreen Forest',
 'Shrub/Scrub',
 'Grassland/Herbaceous',
 'Woody Wetlands',
 'Emergent Herbaceous Wetlands',
 'No Data / Masked']

In [13]:
#Create a vectorized version of your dictionary mapping
def map_names(val):
    return nlcd_legend.get(val, "Unknown")

# Create a vectorized function that can be applied to the entire array of land cover codes to get the corresponding names
v_map_names = np.vectorize(map_names) 

# Create a new DataArray with the string names
cover_names = xr.apply_ufunc(v_map_names, cover)

# Get unique values and their pixel counts
vals, counts = np.unique(cover.values, return_counts=True)

# Build a summary DataFrame
summary = pd.DataFrame({
    "code": vals,
    "count": counts,
    "name": [nlcd_legend.get(v, "Unknown") for v in vals]
})

# Calculate percentage (ignoring 127/No Data in the total pixel count)
total_pixels = summary[summary['code'] != 127]['count'].sum()
summary['percent'] = (summary['count'] / total_pixels) * 100
summary = summary[summary['code'] != 127]

#print(summary.sort_values("percent", ascending=False))

# copy the summary dataframe to a new variable to transpose
lulc = summary.copy()
#remove the following columns from the dataframe: count, code
lulc = lulc.drop(columns=["count", "code"])
#set the name column as the index of the dataframe
lulc.rename(columns={"name": "Basin_Name", 'percent': basin_info['Basin_Name'].values[0]}, inplace=True)
lulc.set_index("Basin_Name", inplace=True)
#transpose the dataframe
lulc = lulc.T
lulc.columns = lulc.columns.str.replace(" ", "_").str.replace("(", "").str.replace(")", "").str.replace("/", "_")

# #combine the basin info dataframe with the lulc dataframe
basin_info.set_index('Basin_Name', inplace=True)
basin_info = pd.concat([basin_info, lulc], axis=1)

OutputFolder = 'files/basin_info'
if not os.path.exists(OutputFolder):
    os.makedirs(OutputFolder)
basin_info.to_csv(f'{OutputFolder}/basin_info_{station_id}.csv')

basin_info


,station_id,Average_Elevation_m,Minimum_Elevation_m,Maximum_Elevation_m,Average_Slope,Area_km2,Open_Water,Perennial_Ice_Snow,"Developed,_Open_Space","Developed,_Low_Intensity","Developed,_Medium_Intensity","Developed,_High_Intensity",Barren_Land_Rock_Sand_Clay,Deciduous_Forest,Evergreen_Forest,Shrub_Scrub,Grassland_Herbaceous,Woody_Wetlands,Emergent_Herbaceous_Wetlands
ColoradoRiverBasin,09010500,3200.7634,2659.6445,3941.0696,0.363304,292.895387,0.076363,3.148496,0.516917,0.076363,0.023496,0.017622,3.724154,0.105733,59.821429,18.708882,9.151786,2.267387,2.361372
